# 04 Dynamic Programming and Sequence Alignment

This notebook introduces dynamic programming for biological sequence comparison.

It supports **Module 06: Dynamic Programming and Sequence Alignment** and connects selected Rosalind problems such as `FIB`, `FIBD`, `LCSQ`, `EDIT`, `EDTA`, `SCSP`, `GLOB`, `LOCA`, `GAFF`, and selected `BA5` Textbook Track problems.

The goal is to show how recurrence relations become reusable table-filling methods for comparing DNA, RNA, and protein sequences.

## Learning Goals

After completing this notebook, a learner should be able to:

- explain why recurrence leads naturally to dynamic programming;
- build dynamic programming tables;
- compute the longest common subsequence;
- compute edit distance;
- reconstruct a simple alignment;
- perform global sequence alignment;
- perform local sequence alignment;
- understand the role of match, mismatch, and gap scores;
- connect alignment scores to biological sequence similarity.

## Connection to Original Rosalind Solutions

This notebook is connected to my original Rosalind solutions preserved under:

- `original_rosalind_tracks/bioinformatics_stronghold/`
- `original_rosalind_tracks/bioinformatics_textbook_track/`

The notebook uses teaching-oriented implementations of dynamic programming and alignment concepts. The original solution files remain the solved-work archive, while this notebook reorganizes selected ideas for explanation, learning, and future reuse.

## 1. Recurrence as the foundation

Many dynamic programming problems start with a recurrence.

A recurrence defines the answer to a larger problem using answers to smaller subproblems.

The Fibonacci recurrence is a simple example:

`F(n) = F(n - 1) + F(n - 2)`

In [ ]:
def fibonacci(n: int) -> int:
    """Return the nth Fibonacci number using dynamic programming style iteration."""
    if n <= 0:
        raise ValueError("n must be positive")

    if n in (1, 2):
        return 1

    dp = [0] * (n + 1)
    dp[1] = 1
    dp[2] = 1

    for i in range(3, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]

    return dp[n]


for n in range(1, 11):
    print(n, fibonacci(n))

## 2. Dynamic programming tables

In sequence problems, we often build a table where each cell stores the answer to a smaller subproblem.

For two sequences `s` and `t`, a table cell `dp[i][j]` may represent the answer for:

- the prefix `s[:i]`
- the prefix `t[:j]`

This idea appears in longest common subsequence, edit distance, and sequence alignment.

## 3. Longest Common Subsequence

This connects to Rosalind problem `LCSQ`.

A subsequence does not need to be contiguous. The longest common subsequence between two strings is the longest sequence of characters that appears in both strings in the same order.

In [ ]:
def lcs_length(s: str, t: str) -> list[list[int]]:
    """Return the dynamic programming table for LCS length."""
    rows = len(s) + 1
    columns = len(t) + 1

    dp = [[0] * columns for _ in range(rows)]

    for i in range(1, rows):
        for j in range(1, columns):
            if s[i - 1] == t[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    return dp


s = "AACCTTGG"
t = "ACACTGTGA"

table = lcs_length(s, t)
table[-1][-1]

## 4. Reconstructing the Longest Common Subsequence

The table gives the length. To recover the actual subsequence, we backtrack through the table.

In [ ]:
def longest_common_subsequence(s: str, t: str) -> str:
    """Return one longest common subsequence of s and t."""
    dp = lcs_length(s, t)

    i = len(s)
    j = len(t)
    result = []

    while i > 0 and j > 0:
        if s[i - 1] == t[j - 1]:
            result.append(s[i - 1])
            i -= 1
            j -= 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            i -= 1
        else:
            j -= 1

    return "".join(reversed(result))


longest_common_subsequence(s, t)

## 5. Edit distance

This connects to Rosalind problem `EDIT`.

Edit distance is the minimum number of operations needed to transform one string into another.

Allowed operations are usually:

- insertion;
- deletion;
- substitution.

In [ ]:
def edit_distance(s: str, t: str) -> int:
    """Return the edit distance between two strings."""
    rows = len(s) + 1
    columns = len(t) + 1

    dp = [[0] * columns for _ in range(rows)]

    for i in range(rows):
        dp[i][0] = i

    for j in range(columns):
        dp[0][j] = j

    for i in range(1, rows):
        for j in range(1, columns):
            substitution_cost = 0 if s[i - 1] == t[j - 1] else 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,                       # deletion
                dp[i][j - 1] + 1,                       # insertion
                dp[i - 1][j - 1] + substitution_cost,   # substitution or match
            )

    return dp[-1][-1]


edit_distance("PLEASANTLY", "MEANLY")

## 6. Edit distance table

Sometimes it is useful to return the full table so we can reconstruct an alignment.

In [ ]:
def edit_distance_table(s: str, t: str) -> list[list[int]]:
    """Return the edit-distance dynamic programming table."""
    rows = len(s) + 1
    columns = len(t) + 1

    dp = [[0] * columns for _ in range(rows)]

    for i in range(rows):
        dp[i][0] = i

    for j in range(columns):
        dp[0][j] = j

    for i in range(1, rows):
        for j in range(1, columns):
            substitution_cost = 0 if s[i - 1] == t[j - 1] else 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + substitution_cost,
            )

    return dp


edit_table = edit_distance_table("GATTACA", "GCATGCU")
edit_table[-1][-1]

## 7. A simple edit alignment

This connects conceptually to Rosalind problem `EDTA`.

The function below reconstructs one possible alignment corresponding to the edit-distance table.

In [ ]:
def edit_alignment(s: str, t: str) -> tuple[str, str]:
    """Return one alignment of s and t based on edit distance."""
    dp = edit_distance_table(s, t)

    i = len(s)
    j = len(t)

    aligned_s = []
    aligned_t = []

    while i > 0 or j > 0:
        if i > 0 and j > 0:
            substitution_cost = 0 if s[i - 1] == t[j - 1] else 1

            if dp[i][j] == dp[i - 1][j - 1] + substitution_cost:
                aligned_s.append(s[i - 1])
                aligned_t.append(t[j - 1])
                i -= 1
                j -= 1
                continue

        if i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            aligned_s.append(s[i - 1])
            aligned_t.append("-")
            i -= 1
        else:
            aligned_s.append("-")
            aligned_t.append(t[j - 1])
            j -= 1

    return "".join(reversed(aligned_s)), "".join(reversed(aligned_t))


alignment = edit_alignment("GATTACA", "GCATGCU")
print(alignment[0])
print(alignment[1])

## 8. Global alignment

This connects to Rosalind problem `GLOB` and Textbook Track problems such as `BA5E`.

Global alignment aligns two full sequences from beginning to end.

The simple scoring scheme below uses:

- match: `+1`
- mismatch: `-1`
- gap: `-1`

In [ ]:
def global_alignment_score(
    s: str,
    t: str,
    match_score: int = 1,
    mismatch_score: int = -1,
    gap_score: int = -1,
) -> int:
    """Return the global alignment score for two sequences."""
    rows = len(s) + 1
    columns = len(t) + 1

    dp = [[0] * columns for _ in range(rows)]

    for i in range(1, rows):
        dp[i][0] = dp[i - 1][0] + gap_score

    for j in range(1, columns):
        dp[0][j] = dp[0][j - 1] + gap_score

    for i in range(1, rows):
        for j in range(1, columns):
            diagonal_score = match_score if s[i - 1] == t[j - 1] else mismatch_score

            dp[i][j] = max(
                dp[i - 1][j - 1] + diagonal_score,
                dp[i - 1][j] + gap_score,
                dp[i][j - 1] + gap_score,
            )

    return dp[-1][-1]


global_alignment_score("GATTACA", "GCATGCU")

## 9. Local alignment

This connects to Rosalind problem `LOCA` and Textbook Track problem `BA5F`.

Local alignment finds the best matching region between two sequences rather than forcing the entire sequences to align.

In [ ]:
def local_alignment_score(
    s: str,
    t: str,
    match_score: int = 2,
    mismatch_score: int = -1,
    gap_score: int = -1,
) -> int:
    """Return the best local alignment score for two sequences."""
    rows = len(s) + 1
    columns = len(t) + 1

    dp = [[0] * columns for _ in range(rows)]
    best_score = 0

    for i in range(1, rows):
        for j in range(1, columns):
            diagonal_score = match_score if s[i - 1] == t[j - 1] else mismatch_score

            dp[i][j] = max(
                0,
                dp[i - 1][j - 1] + diagonal_score,
                dp[i - 1][j] + gap_score,
                dp[i][j - 1] + gap_score,
            )

            best_score = max(best_score, dp[i][j])

    return best_score


local_alignment_score("MEANLY", "PLEASANTLY")

## 10. Alignment scores as sequence similarity features

Alignment scores can become numerical features for downstream analysis.

For example, we can compute pairwise edit distances between multiple sequences and store them in a distance matrix.

In [ ]:
import pandas as pd


def pairwise_edit_distance_matrix(records: dict[str, str]) -> pd.DataFrame:
    """Return a pairwise edit-distance matrix for a dictionary of sequences."""
    ids = list(records.keys())
    matrix = []

    for id_1 in ids:
        row = []
        for id_2 in ids:
            row.append(edit_distance(records[id_1], records[id_2]))
        matrix.append(row)

    return pd.DataFrame(matrix, index=ids, columns=ids)


records = {
    "seq_1": "GATTACA",
    "seq_2": "GCATGCU",
    "seq_3": "GACTATA",
}

pairwise_edit_distance_matrix(records)

## 11. Mini exercise set

Try modifying the functions above to solve these small exercises.

1. Modify `global_alignment_score()` so that mismatch and gap penalties can be different.
2. Write a function that returns the full global alignment, not only the score.
3. Modify `local_alignment_score()` so it also returns the best local alignment substring.
4. Build a pairwise global-alignment score matrix for a dictionary of sequences.
5. Compare edit distance and LCS length for the same pair of sequences.
6. Explain why local alignment is more appropriate than global alignment for comparing sequence regions.

## Summary

This notebook introduced dynamic programming as a central tool for computational biology.

| Concept | Bioinformatics Use |
|---|---|
| Recurrence | foundation of dynamic programming |
| DP table | stores solutions to smaller sequence problems |
| LCS | subsequence similarity |
| Edit distance | sequence difference |
| Alignment reconstruction | interpretable sequence comparison |
| Global alignment | full-sequence comparison |
| Local alignment | best-region comparison |
| Distance matrix | ML-ready similarity or distance representation |

These ideas prepare the learner for later work in string matching, phylogenetics, proteomics, and ML-ready biological sequence representation.